In [1]:
'''

single_point_failure_postprocess.py

    This script is used to conduct postprocessing on the results from single-point
    failure analyses of nodes and edges (../linux/outputs/<node/edge>_impact_assessment)

'''

print('------------------------------------------------------------')
print('Run: single_point_failure_postprocess.py')
print('')

import os
import pandas as pd

# ---------------
# GLOBAL VARS

output_path     = '../data/single_point_failure/'
node_path       = '../linux/outputs/node_impact_assessment/' 
edge_path       = '../linux/outputs/edge_impact_assessment/'

# ---------------
# NODES

# read results and merge into one df
merged_results = []
for i in os.listdir(node_path):
    if '.csv' in i:
        d = pd.read_csv(node_path + i)
        prev_cols = list(d.columns)
        d['batch_number'] = int(i.split('_')[4])
        d = d[ ['batch_number'] + prev_cols ]
        merged_results.append(d)

print('merged node results')

# concat
merged_results = pd.concat(merged_results,ignore_index=True)
merged_results = merged_results.sort_values(['batch_number','iteration_number']).reset_index(drop=True)
# remove duplicates
merged_processed = merged_results.copy()
prev_cols =  merged_processed.columns

merged_processed = merged_processed.groupby(['attacked_node_id',
                                             'affected_node_id'],as_index=False,dropna=False).first()

#merged_processed = merged_processed.drop(['batch_number','iteration_number'],axis=1)
merged_processed = merged_processed.sort_values(['batch_number','iteration_number'])
merged_processed = merged_processed[prev_cols].reset_index(drop=True)

print('removed duplicates')

# save
merged_processed.to_csv(output_path + 'single_point_failure_results_nodes.csv',index=False)
print('saved merged node results')

# ---------------
# EDGES

# read results and merge into one df
merged_results = []
for i in os.listdir(edge_path):
    if '.csv' in i:
        d = pd.read_csv(edge_path + i)
        prev_cols = list(d.columns)
        d['batch_number'] = int(i.split('.')[0].split('_')[4])
        d = d[ ['batch_number'] + prev_cols ]
        merged_results.append(d)
        
print('merged edge results')

# concat
merged_results = pd.concat(merged_results,ignore_index=True)
merged_results = merged_results.sort_values(['batch_number','iteration_number']).reset_index(drop=True)
# remove duplicates
merged_processed = merged_results.copy()
prev_cols =  merged_processed.columns

merged_processed = merged_processed.groupby(['attacked_edge_id',
                                             'affected_node_id'],as_index=False,dropna=False).first()

#merged_processed = merged_processed.drop(['batch_number','iteration_number'],axis=1)
merged_processed = merged_processed.sort_values(['batch_number','iteration_number'])
merged_processed = merged_processed[prev_cols].reset_index(drop=True)

print('removed duplicates')

# save
merged_processed.to_csv(output_path + 'single_point_failure_results_edges.csv',index=False)

print('saved merged edge results')
print('done')

--------------------
Run: single_point_failure_postprocess.py

merged node results
removed duplicates
saved merged node results


ValueError: invalid literal for int() with base 10: '12.csv'

In [9]:
for i in os.listdir(node_path):
    if '.csv' in i:
        print(i)
        #print(int(i.split('_')[4]))
        #d = d[ ['batch_number'] + prev_cols ]
        #merged_results.append(d)

node_impact_assessment_batch_10_iteration_500.csv
node_impact_assessment_batch_17_iteration_2000.csv
node_impact_assessment_batch_4_iteration_2500.csv
node_impact_assessment_batch_9_iteration_2500.csv
node_impact_assessment_batch_9_iteration_500.csv
node_impact_assessment_batch_13_iteration_1000.csv
node_impact_assessment_batch_2_iteration_2000.csv
node_impact_assessment_batch_15_iteration_500.csv
node_impact_assessment_batch_18_iteration_1500.csv
node_impact_assessment_batch_15_iteration_1500.csv
node_impact_assessment_batch_6_iteration_1000.csv
node_impact_assessment_batch_11_iteration_2500.csv
node_impact_assessment_batch_5_iteration_1000.csv
node_impact_assessment_batch_11_iteration_500.csv
node_impact_assessment_batch_8_iteration_1000.csv
node_impact_assessment_batch_12_iteration_2500.csv
node_impact_assessment_batch_1_iteration_2000.csv
node_impact_assessment_batch_8_iteration_500.csv
node_impact_assessment_batch_16_iteration_1500.csv
node_impact_assessment_batch_14_iteration_500

In [12]:
for i in os.listdir(edge_path):
    if '.csv' in i:
        i = i.split('.')[0].split('_')[4]
        print(i)
        #print(int(i.split('_')[4]))
        #d = d[ ['batch_number'] + prev_cols ]
        #merged_results.append(d)

12
13
11
10
14
15
17
16
8
9
1
2
3
7
6
4
5
18
19


In [8]:
i.split('_')

['edge', 'impact', 'assessment', 'batch', '12.csv']

In [5]:
'''

map_water_assets.py

    This script creates mapping between the JEM nodal file and the nismod.jamaica.water 
    sector assets. Specifically, it links each energy-consumptive water asset to the nearest
    utility pole or substation in the energy sector nodal file. 
    

'''

import geopandas as gpd
from shapely.ops import nearest_points
from tqdm import tqdm

print('loading data...')
elec_nodes = gpd.read_file('../data/spatial/infrasim-network/nodes.shp')
water_nodes = gpd.read_file('../data/water/merged_water_assets.shp')
print('done')

water_id_col = 'node_id'
output_col_name_water = 'water_node_id'
output_col_name_elect = 'elec_node_id'

# index nodes
print('indexing node data...')
# gpd1 = elec_nodes.loc[elec_nodes.subtype.isin(['pole','substation'])].reset_index(drop=True).copy()
gpd1 = elec_nodes.loc[elec_nodes.asset_type.isin(['sink'])].reset_index(drop=True).copy()
gpd2 = water_nodes.loc[water_nodes.linkage == 'True'].reset_index(drop=True).copy()

# form unary union
pts3 = gpd1.geometry.unary_union
# init elec_asset column
gpd2[output_col_name_elect] = ''

# loop through each water asset
print('looping across water nodes to map energy assets...')
for i, row in gpd2.iterrows():
    print(i)
    nearest_elec_asset = nearest_points(row.geometry, pts3)[1]
    gpd2.loc[i,output_col_name_elect] = gpd1.loc[gpd1.geometry.within(nearest_elec_asset),'id'].values[0]
# append water nodes
water_nodes[output_col_name_elect] = water_nodes[water_id_col].map(\
                gpd2.set_index(water_id_col)[output_col_name_elect].to_dict())
print('done')

#rename columsn
water_nodes[output_col_name_water] = water_nodes[water_id_col]

# save to shapefile
print('saving file...')
water_nodes.to_file(driver='ESRI Shapefile',filename='../data/water/mapped_water_assets.shp')
# save to csv
water_nodes[[output_col_name_water,output_col_name_elect]].to_csv('../data/water/mapped_water_assets.csv',index=False)
print('done')
print('files saved to: ../data/water/mapped_water_assets.shp')

loading data...
done
indexing node data...
looping across water nodes to map energy assets...
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68


KeyboardInterrupt: 

,id,asset_type,subtype,capacity,uc_min,uc_max,uc_avg,uc_uom,cost_min,cost_max,cost_avg,cost_uom,degree,population,ei,ei_uom,parish,title,source,geometry
0,node_1,source,solar,20.0,2517638,3489415,3003527,USD/MW,50352760.0,69788300.0,60070540.0,$US,2,0.000000,0.083847,KW/person,Clarendon,Content,Majid,POINT (717192.535 643760.127)
1,node_2,junction,substation,8.4,10700,24000,17350,USD/MW,89880.0,201600.0,145740.0,$US,2,0.000000,0.083847,KW/person,Clarendon,WI PULP & PAPER,Edson,POINT (734584.350 638724.566)
2,node_3,junction,substation,96.6,10700,24000,17350,USD/MW,1033620.0,2318400.0,1676010.0,$US,16,0.000000,0.083847,KW/person,Clarendon,PARNASSUS,Edson,POINT (722299.190 642200.102)
3,node_4,junction,substation,48.3,10700,24000,17350,USD/MW,516810.0,1159200.0,838005.0,$US,2,0.000000,0.083847,KW/person,Clarendon,MAY PEN,Edson,POINT (724027.959 647346.567)
4,node_5,junction,substation,48.3,10700,24000,17350,USD/MW,516810.0,1159200.0,838005.0,$US,8,0.000000,0.083847,KW/person,Clarendon,MONYMUSK,Edson,POINT (723626.723 629267.700)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44030,node_42966,sink,demand,16.8,10700,24000,17350,USD/MW,179760.0,403200.0,291480.0,$US,2,187.017997,0.114055,KW/person,Westmoreland,None,None,POINT (629913.964 676509.534)
44031,node_42967,sink,demand,16.8,10700,24000,17350,USD/MW,179760.0,403200.0,291480.0,$US,2,0.040143,0.114055,KW/person,Westmoreland,None,None,POINT (622952.452 675772.820)
44032,node_42968,junction,pole,16.8,10700,24000,17350,USD/MW,179760.0,403200.0,291480.0,$US,4,0.000000,0.114055,KW/person,Westmoreland,None,None,POINT (623278.970 676101.853)
44033,node_42969,junction,pole,16.8,10700,24000,17350,USD/MW,179760.0,403200.0,291480.0,$US,6,0.000000,0.114055,KW/person,Westmoreland,None,None,POINT (623390.150 676218.551)
